# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

## Limpieza y formateo previos

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
raw = pd.read_csv(url)
print(raw.shape)
raw.head()

(10910, 26)


,Unnamed: 0,Customer,State,Customer Lifetime Value,Response,Coverage,Education,Effective To Date,EmploymentStatus,Gender,...,Number of Open Complaints,Number of Policies,Policy Type,Policy,Renew Offer Type,Sales Channel,Total Claim Amount,Vehicle Class,Vehicle Size,Vehicle Type
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2/18/11,Employed,M,...,0.0,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,NaN
1,1,KX64629,California,2228.525238,No,Basic,College,1/18/11,Unemployed,F,...,0.0,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,NaN
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2/10/11,Employed,M,...,0.0,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,1/17/11,Medical Leave,F,...,NaN,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,NaN


In [2]:
# Nombres de columna en snake_case y fuera "Unnamed: 0" (es el índice antiguo guardado como columna)
df = raw.drop(columns="Unnamed: 0")
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")

print(df.isnull().sum()[df.isnull().sum() > 0], "\n")
print("Filas duplicadas exactas:", df.duplicated().sum())
print("IDs de cliente repetidos:", df["customer"].duplicated().sum())

state                         631
response                      631
months_since_last_claim       633
number_of_open_complaints     633
vehicle_class                 622
vehicle_size                  622
vehicle_type                 5482
dtype: int64 

Filas duplicadas exactas: 37
IDs de cliente repetidos: 1776


Hay **1776 IDs de cliente repetidos** pero solo 37 filas idénticas. Miramos un ejemplo:

In [3]:
df[df["customer"] == "AA10041"].T

,3960,8731,10376
customer,AA10041,AA10041,AA10041
state,NaN,California,California
customer_lifetime_value,7901.744238,7901.744238,7901.744238
response,NaN,No,No
coverage,Basic,Basic,Basic
education,High School or Below,High School or Below,High School or Below
effective_to_date,2/2/11,2/2/11,2/2/11
employmentstatus,Unemployed,Unemployed,Unemployed
gender,M,M,M
income,0,0,0


Las filas repetidas son **el mismo cliente copiado con algunos campos borrados** (NaN en `state`, `response`, `vehicle_class`...). Antes de fusionarlas comprobamos que las copias de un mismo cliente **nunca se contradicen** (que ninguna columna tenga dos valores distintos no nulos):

In [4]:
conflicts = (df.groupby("customer").nunique() > 1).sum()
conflicts[conflicts > 0]  # vacío = no hay contradicciones

Series([], dtype: int64)

**Estrategia:** no hay contradicciones, así que agrupamos por cliente y nos quedamos con el primer valor **no nulo** de cada columna (`groupby().first()` ignora los NaN). Con esto:
- cada cliente aparece una sola vez, lo que es clave para contar clientes y pólizas sin inflar los números;
- se rellenan casi todos los nulos con datos **reales** de las otras copias, sin inventar nada.

In [5]:
df = df.groupby("customer", as_index=False, sort=False).first()
print(df.shape)
df.isnull().sum()[df.isnull().sum() > 0]

(9134, 25)


vehicle_type    4173
dtype: int64

Solo quedan nulos en `vehicle_type`, que solo tiene el valor `"A"` o vacío. No sabemos qué significa el vacío, así que lo marcamos como `"Unknown"` en vez de inventar un valor.

Además convertimos `effective_to_date` a fecha y sacamos el mes, que lo necesitaremos en el bonus.

In [6]:
df["vehicle_type"] = df["vehicle_type"].fillna("Unknown")
df["effective_to_date"] = pd.to_datetime(df["effective_to_date"], format="%m/%d/%y")
df["month"] = df["effective_to_date"].dt.month

print("Nulos:", df.isnull().sum().sum(), "| IDs repetidos:", df["customer"].duplicated().sum())
df.dtypes

Nulos:

 0 | IDs repetidos: 0


customer                                    str
state                                       str
customer_lifetime_value                 float64
response                                    str
coverage                                    str
education                                   str
effective_to_date                datetime64[us]
employmentstatus                            str
gender                                      str
income                                    int64
location_code                               str
marital_status                              str
monthly_premium_auto                      int64
months_since_last_claim                 float64
months_since_policy_inception             int64
number_of_open_complaints               float64
number_of_policies                        int64
policy_type                                 str
policy                                      str
renew_offer_type                            str
sales_channel                           

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [7]:
low_claim_yes = df[(df["total_claim_amount"] < 1000) & (df["response"] == "Yes")]
print(low_claim_yes.shape)
low_claim_yes.head()

(1248, 26)


,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,income,...,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type,month
3,XL78013,Oregon,22332.439460,Yes,Extended,College,2011-01-11,Employed,M,49078,...,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A,1
8,FM55990,California,5989.773931,Yes,Premium,College,2011-01-19,Employed,M,66839,...,1,Personal Auto,Personal L1,Offer2,Branch,739.200000,Sports Car,Medsize,Unknown,1
15,CW49887,California,4626.801093,Yes,Basic,Master,2011-01-16,Employed,F,79487,...,1,Special Auto,Special L1,Offer2,Branch,547.200000,SUV,Medsize,Unknown,1
19,NJ54277,California,3746.751625,Yes,Extended,College,2011-02-26,Employed,F,41479,...,1,Personal Auto,Personal L2,Offer2,Call Center,19.575683,Two-Door Car,Large,A,2
27,MQ68407,Oregon,4376.363592,Yes,Premium,Bachelor,2011-02-28,Employed,F,63774,...,1,Personal Auto,Personal L3,Offer2,Agent,60.036683,Four-Door Car,Medsize,A,2


2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [8]:
yes = df[df["response"] == "Yes"]

segments = yes.groupby(["policy_type", "gender"]).agg(
    customers=("customer", "count"),
    avg_monthly_premium=("monthly_premium_auto", "mean"),
    avg_clv=("customer_lifetime_value", "mean"),
    avg_claim=("total_claim_amount", "mean"),
).round(2)

# Ratio reclamación / prima: cuántas primas mensuales "se come" cada cliente en siniestros.
# Cuanto MÁS BAJO, más rentable y menos arriesgado es el segmento.
segments["claim_to_premium"] = (segments["avg_claim"] / segments["avg_monthly_premium"]).round(2)
segments.sort_values("claim_to_premium")

customers  avg_monthly_premium  avg_clv  avg_claim  \
policy_type    gender                                                       
Corporate Auto M             139                91.67  7833.14     412.76   
Personal Auto  F             479                98.82  8262.72     454.09   
Corporate Auto F             149                93.67  7674.45     431.48   
Special Auto   M              30                86.37  8608.17     414.80   
               F              32                92.62  8122.33     455.65   
Personal Auto  M             479                90.92  7444.40     453.60   

                       claim_to_premium  
policy_type    gender                    
Corporate Auto M                   4.50  
Personal Auto  F                   4.60  
Corporate Auto F                   4.61  
Special Auto   M                   4.80  
               F                   4.92  
Personal Auto  M                   4.99

In [9]:
# Para comparar: ¿se parecen los que dijeron "Yes" a los que dijeron "No"?
df.groupby("response")[["monthly_premium_auto", "customer_lifetime_value", "total_claim_amount"]].mean().round(2)

,monthly_premium_auto,customer_lifetime_value,total_claim_amount
response,,,
No,93.07,8030.02,432.08
Yes,94.14,7854.87,446.08


**Insights:**
- **Corporate Auto** (hombres y mujeres) es el segmento **más rentable y de menor riesgo**: tiene el ratio reclamación/prima más bajo (~4,5), es decir, reclama menos en relación con lo que paga.
- **Hombres con Personal Auto** es el segmento **menos rentable**: pagan la prima más baja de los segmentos grandes (~91 $) y reclaman lo mismo que las mujeres (~454 $), lo que da el ratio más alto (~5).
- **Mujeres con Personal Auto** pagan la prima más alta (~99 $) y tienen un CLV alto (~8.260 $). Como además es un segmento grande (479 clientes), es el **más valioso en volumen**.
- Special Auto tiene el CLV más alto en hombres (~8.600 $), pero con solo ~30 clientes por grupo no se puede sacar una conclusión fiable.
- Los clientes que responden "Yes" son muy parecidos a los que responden "No": la campaña no atrae clientes especialmente mejores ni peores.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [10]:
customers_by_state = df.groupby("state")["customer"].nunique().sort_values(ascending=False)
customers_by_state

state
California    3150
Oregon        2601
Arizona       1703
Nevada         882
Washington     798
Name: customer, dtype: int64

In [11]:
customers_by_state[customers_by_state > 500]

state
California    3150
Oregon        2601
Arizona       1703
Nevada         882
Washington     798
Name: customer, dtype: int64

Todos los estados superan los 500 clientes, así que el filtro no elimina ninguno. California concentra ~35 % de los clientes y Washington es el más pequeño (~800).

*Nota: uso `nunique()` y no `count()`. Con el dataset sin deduplicar, `count()` habría contado varias veces al mismo cliente.*

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [12]:
clv_stats = df.groupby(["education", "gender"])["customer_lifetime_value"].agg(["max", "min", "median"]).round(2)
clv_stats

max      min   median
education            gender                            
Bachelor             F       73225.96  1904.00  5678.05
                     M       67907.27  1898.01  5555.83
College              F       61850.19  1898.68  5621.79
                     M       61134.68  1918.12  5989.77
Doctor               F       44856.11  2395.57  5332.46
                     M       32677.34  2267.60  5620.59
High School or Below F       55277.45  2144.92  6044.02
                     M       83325.38  1940.98  6176.70
Master               F       51016.07  2417.78  5801.13
                     M       50568.26  2272.31  5617.96

**Conclusiones:**
- La **mediana** es muy parecida en todos los grupos (~5.300–6.200 $): ni la educación ni el género cambian mucho el valor de un cliente típico.
- **High School or Below** tiene la mediana más alta en hombres y mujeres (~6.000–6.200 $), y el cliente más valioso del dataset (hombre, ~83.300 $).
- **Doctor** tiene la mediana más baja (~5.300–5.600 $) y también los máximos más bajos. Más estudios no implica más valor de cliente.
- Los máximos (32.000–83.000 $) están muy lejos de las medianas: el CLV tiene **outliers muy fuertes**. Por eso aquí es mejor comparar la mediana que la media.
- Los mínimos son casi iguales en todos los grupos (~1.900–2.400 $), lo que parece un valor mínimo común a todas las pólizas.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

In [13]:
# Cada fila del dataset (ya deduplicado) es una póliza
policies_state_month = df.pivot_table(index="state", columns="month", values="customer", aggfunc="count")
policies_state_month.columns = ["January", "February"]
policies_state_month

,January,February
state,,
Arizona,899,804
California,1695,1455
Nevada,494,388
Oregon,1396,1205
Washington,414,384


En todos los estados se vendieron más pólizas en enero que en febrero. Febrero tiene menos días, pero la caída (~13 %) es mayor que la diferencia de días (~10 %).

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

In [14]:
# 1) Agrupar por estado y mes y contar pólizas
by_state_month = df.groupby(["state", "month"]).size().reset_index(name="policies_sold")

# 2) Top 3 estados por total de pólizas
top3_states = by_state_month.groupby("state")["policies_sold"].sum().nlargest(3).index
print(list(top3_states))

# 3) Filtrar solo esos estados y ordenar de mayor a menor
top3 = (
    by_state_month[by_state_month["state"].isin(top3_states)]
    .sort_values("policies_sold", ascending=False)
    .reset_index(drop=True)
)
top3

['California', 'Oregon', 'Arizona']


,state,month,policies_sold
0,California,1,1695
1,California,2,1455
2,Oregon,1,1396
3,Oregon,2,1205
4,Arizona,1,899
5,Arizona,2,804


7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

In [15]:
# Tabla ancha: número de clientes que respondieron Yes / No por canal
response_by_channel = df.pivot_table(index="sales_channel", columns="response", values="customer", aggfunc="count")
response_by_channel["response_rate_%"] = (
    response_by_channel["Yes"] / (response_by_channel["Yes"] + response_by_channel["No"]) * 100
).round(2)
response_by_channel.sort_values("response_rate_%", ascending=False)

response,No,Yes,response_rate_%
sales_channel,,,
Agent,2811,666,19.15
Web,1169,156,11.77
Branch,2273,294,11.45
Call Center,1573,192,10.88


In [16]:
# Con melt la pasamos a formato largo: una fila = (canal, respuesta, nº clientes)
response_long = response_by_channel[["No", "Yes"]].reset_index().melt(
    id_vars="sales_channel", var_name="response", value_name="customers"
)
response_long["pct_of_channel"] = (
    response_long["customers"] / response_long.groupby("sales_channel")["customers"].transform("sum") * 100
).round(2)
response_long.sort_values(["sales_channel", "response"])

,sales_channel,response,customers,pct_of_channel
0,Agent,No,2811,80.85
4,Agent,Yes,666,19.15
1,Branch,No,2273,88.55
5,Branch,Yes,294,11.45
2,Call Center,No,1573,89.12
6,Call Center,Yes,192,10.88
3,Web,No,1169,88.23
7,Web,Yes,156,11.77


**Insights:**
- **Agent** tiene una tasa de respuesta del **~19 %**, casi el **doble** que el resto de canales (~11–12 %).
- Branch, Call Center y Web funcionan prácticamente igual.
- Recomendación: el contacto personal de un agente convierte mucho mejor. Si el objetivo es que los clientes respondan a la campaña, conviene reforzar el canal Agent o estudiar qué hacen los agentes para aplicarlo en el resto de canales.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9